In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("fashion-mnist_train.csv")
df_test=pd.read_csv("fashion-mnist_test.csv")

In [3]:
#Training-Validation split
df1=df.copy()
df1=df.sample(frac=1,random_state=42)
train_size=int(0.8*len(df))
df_train=df1[:train_size].copy()
df_val=df1[train_size:].copy()

In [4]:
#Partioning the dataset into features and labels
y_train=df_train['label']
y_val=df_val["label"]
y_test=df_test["label"]
X_train=df_train.drop("label",axis=1)
X_val=df_val.drop("label",axis=1)
X_test=df_test.drop("label",axis=1)

In [5]:
#Min-Max Scaling
num_columns = [ col for col in df.columns if col != "label"]
X_train[num_columns]=(X_train[num_columns]/255)
X_val[num_columns]=(X_val[num_columns]/255)
X_test[num_columns]=(X_test[num_columns]/255)

In [6]:
#Creating Activation functions using Numpy
def stable_softmax(matrix):
    numerator=np.exp(matrix-np.max(matrix,axis=1,keepdims=True))
    transformed=(numerator)/(np.sum((numerator),axis=1,keepdims=True))
    return transformed
def relu(matrix):
    matrix_0=np.zeros(matrix.shape)
    return np.maximum(matrix,matrix_0)  

In [7]:
#Loss Function for Backpropagation
def cross_entropy_loss(y_true,y_probab):
    a=1e-6
    loss=-np.mean(np.sum(y_true*np.log(y_probab+a),axis=1))
    return loss 

In [8]:
#Forward Pass for a Variable Layer Neural Network
def forward_pass(dataset,weights,biases):
    Z=list()
    A=list()
    n=len(weights)
    for weight,bias in zip(weights,biases):
      Z.append((dataset@weight)+bias)
      if n>1:
          A.append(relu(Z[-1]))
      else:
          A.append(stable_softmax(Z[-1]))
      dataset=A[-1]
      n-=1
    output_probab=A[-1]
    return Z,A,output_probab

In [9]:
#BackPropagation for a Variable Layer Neural Network where Z and A are the weighted sums and activations of the neurons respectively
def backward_pass(weights,output_probab,Z,A,dataset,y_train):
    dZ=list()
    dA=list()
    dW=list()
    db=list()
    n=len(dataset)
    dZ.append(output_probab-y_train)
    dW.append((A[-2].T@dZ[-1])/n)
    db.append(np.sum(dZ[-1],axis=0,keepdims=True)/n)
    for i in range(len(weights)-2):
        dA.append(dZ[-1]@weights[-1-i].T)
        dZ_new=dA[-1]*(Z[len(weights)-2-i]>0)
        dW.append((A[-3-i].T@dZ_new)/n)
        db.append(np.sum(dZ_new,axis=0,keepdims=True)/n)
        dZ.append(dZ_new)
    dA_1=dZ[-1]@weights[1].T
    dZ_1=dA_1*(Z[0]>0)
    dW.append((dataset.T@dZ_1)/n)
    db.append(np.sum(dZ_1,axis=0,keepdims=True)/n)
    return dW,db

In [10]:
#Update function for a Variable Layer Neural Network consisting of Adam optimizer
def update(learning_rate,dW,db,weights,biases,beta,m,alpha,v,epoch):
    for i in range(len(weights)):
        m[i]=m[i]*beta+(1-beta)*dW[-1-i]           #Momentum
        v[i]=v[i]*alpha +(1-alpha)*dW[-1-i]**2     #RMSProp
        m_bias_correction=m[i]/(1-beta**(epoch+1))
        v_bias_correction=v[i]/(1-alpha**(epoch+1))
        epsilon=1e-6
        weights[i]-=learning_rate*(m_bias_correction)/np.sqrt(v_bias_correction+epsilon)   #Adam Optimizer
        biases[i]-=learning_rate*db[-1-i]
    return weights,biases,m,v

In [11]:
def training(dataset,X_val,layers,y_train,y_val,n_epochs,learning_rate,beta,alpha):
    #Using Kaiming for weights initilizations as we use ReLU activation
    np.random.seed(42)
    weights=list()
    biases=list()
    losses=list()
    val_losses=list()
    for i in range((len(layers)-1)):
        weights.append(np.random.randn(layers[i],layers[i+1])*np.sqrt(2/layers[i]))
        biases.append(np.zeros((1,layers[i+1])))
    m=list()
    v=list()
    for i in range(len(weights)):
        m.append(np.zeros_like(weights[i]))
        v.append(np.zeros_like(weights[i]))
    for epoch in range(n_epochs):
        Z,A,output_probab=forward_pass(dataset,weights,biases)
        dW,db=backward_pass(weights,output_probab,Z,A,dataset,y_train)
        weights,biases,m,v=update(learning_rate,dW,db,weights,biases,beta,m,alpha,v,epoch)
        loss=cross_entropy_loss(y_train,output_probab)
        losses.append(loss)

        #Calculating validation loss at each steo to see if there is overfitting or not, later by comapring it with training losses
        *_,val_probab=forward_pass(X_val,weights,biases)
        val_loss=cross_entropy_loss(y_val,val_probab)
        val_losses.append(val_loss)
        if epoch>0 and abs(prev_loss-loss)<1e-5:
          print(f"Converged at epoch:{epoch}")
          break
        prev_loss=loss
    return weights,biases,losses,val_losses

In [12]:
import time
import tracemalloc

In [ ]:
X_train_num=np.array(X_train)
X_val_num=np.array(X_val)
y_train_num=np.array(y_train)
y_val_num=np.array(y_val)
#Training of the neural network
layers=[784,128,64,10]                         #Variable Layer Neural Network
start=time.time()
tracemalloc.start()
weights,biases,losses,val_losses=training(X_train_num,X_val_num,layers,np.eye(10)[y_train_num],np.eye(10)[y_val_num],n_epochs=10000,learning_rate=0.01,beta=0.9,alpha=0.999)
current,max_memory_used=tracemalloc.get_traced_memory()
tracemalloc.stop()
end=time.time()

In [ ]:
#Training-Validation Loss Curve
import matplotlib.pyplot as plt
plt.plot(losses,label="Train Loss")
plt.plot(val_losses,label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Train vs Validation Loss(Numpy)")
plt.legend()
plt.show()


In [ ]:
#Convergence Plot
import matplotlib.pyplot as plt
plt.plot(losses)
plt.xlabel("Epochs")
plt.ylabel("Cross-Entropy Loss")
plt.title("Convergence Curve(Numpy)")
plt.show()

In [ ]:
#Metrics

def accuracy(y_true,y_probab):
    y_preds=np.argmax(y_probab,axis=1)
    return np.mean(y_true==y_preds)
def confusion_matrix(y_true,y_probab):
  y_preds=np.argmax(y_probab,axis=1)
  confusion_matrix=np.zeros((10,10))
  for i in range(len(y_true)):
      confusion_matrix[y_true[i],y_preds[i]]+=1
  return confusion_matrix
def plot_confusion_matrix(confusion_matrix):
    import seaborn as sns
    import matplotlib.pyplot as plt
    class_names=["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankleboot"] 
    df_cm=pd.DataFrame(confusion_matrix,index=class_names,columns=class_names).astype(int)
    plt.figure(figsize=(10,7))
    ax=sns.heatmap(df_cm,annot=True,cmap="Blues",fmt="d")
    ax.set_xlabel("Predicted Value",labelpad=12)
    ax.set_ylabel("Actual Value",labelpad=12)
    plt.show()

def class_accuracy(confusion_matrix):
    class_names=["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankleboot"] 
    per_class_acc=confusion_matrix.diagonal()/confusion_matrix.sum(axis=1)
    df=pd.DataFrame({"Classes":class_names,"Accuracy(%)":np.round(per_class_acc*100,2)})
    return df

In [ ]:
#Validation:
X_val_num=np.array(X_val)
y_val_num=np.array(y_val)
*_,val_probab=forward_pass(X_val_num,weights,biases)

#Outcomes:
cm=confusion_matrix(y_val_num,val_probab)
print(f"Accuracy:{accuracy(y_val_num,val_probab)}")
print(f"Convergence Time in training:{end-start:.2f} seconds")
print(f"Memory Usage in training:{max_memory_used/1024**2:.2f} MB")
plot_confusion_matrix(cm) 

In [ ]:
#Test set
X_test_num=np.array(X_test)
y_test_num=np.array(y_test)
*_,test_probab=forward_pass(X_test_num,weights,biases)

#Outcomes:
cm=confusion_matrix(y_test_num,test_probab)
print(f"Accuracy:{accuracy(y_test_num,test_probab)}")
print(f"Convergence Time in training:{end-start:.2f} seconds")
print(f"Memory Usage in training:{max_memory_used/1024**2:.2f} MB")
plot_confusion_matrix(cm)

In [ ]:
#Checking the accuracy of model per class
class_accuracy_test=class_accuracy(cm)
class_accuracy_test